In [ ]:
# VIDVRD → STTran training (VIDVRD head) — Colab notebook
#
# Prereq: run `Definitive_test_VIDVRD.ipynb` successfully at least once.

# 0) Mount Drive (zip + outputs)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1) Clone repo (edit URL if you use a fork)
%cd /content
!rm -rf STTran
!git clone <YOUR_GITHUB_REPO_URL> STTran
%cd /content/STTran


In [ ]:
# 2) Setup (idempotent; reruns should be fast)
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u setup_colab.py --colab


In [ ]:
# 3) Unzip VIDVRD zip from Drive → local /content
%%bash
set -euo pipefail
set -x

ZIP="/content/drive/MyDrive/vidvrd-dataset_480.zip"  # <-- adjust if needed
OUT="/content/vidvrd-dataset_480"

if [ -d "$OUT/test_frames_480" ] && [ -d "$OUT/test_480" ] && \
   [ -d "$OUT/train_frames_480" ] && [ -d "$OUT/train_480" ]; then
  echo "[skip] already unzipped at $OUT"
  exit 0
fi

rm -rf "$OUT" /content/VIDVRD-DATASET_480 /content/VIDVRD
unzip -q "$ZIP" -d /content

LOCAL_ROOT=""
for CAND in \
  /content/vidvrd-dataset_480 \
  /content/VIDVRD-DATASET_480 \
  /content/VIDVRD-DATASET_480/VIDVRD-DATASET_480 \
  /content/VIDVRD/VIDVRD-DATASET_480 \
  ; do
  if [ -d "$CAND/train_frames_480" ] && [ -d "$CAND/train_480" ]; then
    LOCAL_ROOT="$CAND"
    break
  fi
done

if [ -z "$LOCAL_ROOT" ]; then
  echo "[debug] /content listing:" >&2
  ls -lah /content | head -n 200 >&2
  echo "[error] Could not find extracted VIDVRD dataset root. Check ZIP structure." >&2
  exit 2
fi

if [ "$LOCAL_ROOT" != "$OUT" ]; then
  rm -rf "$OUT"
  mv "$LOCAL_ROOT" "$OUT"
fi

echo "[ok] dataset_root=$OUT"


In [ ]:
# 4) Build stable vocab once (train split)
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/vidvrd-dataset_480" \
  --vocab_scan_dir "/content/vidvrd-dataset_480/train_480" \
  --save_vocab_json "/content/vidvrd_vocab.json" \
  --video_id "ILSVRC2015_train_00008004" \
  --expected_hw "480,854" \
  --max_frames 4 \
  --no_forward \
  --num_predicates 0

ls -lah /content/vidvrd_vocab.json


In [ ]:
# 5) Train (tiny debug run)
#
# This trains ONLY the VIDVRD head by default. It iterates `--max_videos` videos per epoch.
# Outputs go to Drive.
%cd /content/STTran
!python -u colab/vidvrd_train_colab.py \
  --dataset_root "/content/vidvrd-dataset_480" \
  --vocab_json "/content/vidvrd_vocab.json" \
  --split train \
  --max_videos 2 \
  --max_frames 32 \
  --epochs 1 \
  --lr 1e-3 \
  --out_dir "/content/drive/MyDrive/vidvrd_runs/exp_real_debug"
